# Predicting the 3 month CLV with Python

In [ ]:
# Loading the data
import pandas as pd
import janitor
df = pd.read_excel('../../data/raw/Online Retail.xlsx', sheet_name='Online Retail').clean_names(case_type='snake')

In [ ]:
df.shape

In [ ]:
df.head()

# Data Clean-Up

#### - Negative Quantity

In [ ]:
df.loc[df['quantity'] <= 0].shape

In [ ]:
df.shape

In [ ]:
df = df.loc[df['quantity'] > 0]

In [ ]:
df.shape

#### - Missing CustomerID

In [ ]:
pd.isnull(df['customer_id']).sum()

In [ ]:
df.shape

In [ ]:
df = df[pd.notnull(df['customer_id'])]

In [ ]:
df.shape

In [ ]:
df.head()

#### - Excluding Incomplete Month

In [ ]:
print('Date Range: %s ~ %s' % (df['invoice_date'].min(), df['invoice_date'].max()))

In [ ]:
df.loc[df['invoice_date'] >= '2011-12-01'].shape

In [ ]:
df.shape

In [ ]:
df = df.loc[df['invoice_date'] < '2011-12-01']

In [ ]:
df.shape

#### - Total Sales

In [ ]:
df['sales'] = df['quantity'] * df['unit_price']

In [ ]:
df.head()

#### - Per Order Data

In [ ]:
orders_df = df.groupby(['customer_id', 'invoice_no']).agg({
    'sales': sum,
    'invoice_date': max
})

In [ ]:
orders_df

# Data Analysis

In [ ]:
def groupby_mean(x):
    return x.mean()

def groupby_count(x):
    return x.count()

def purchase_duration(x):
    return (x.max() - x.min()).days

def avg_frequency(x):
    return (x.max() - x.min()).days/x.count()

groupby_mean.__name__ = 'avg'
groupby_count.__name__ = 'count'
purchase_duration.__name__ = 'purchase_duration'
avg_frequency.__name__ = 'purchase_frequency'

In [ ]:
summary_df = orders_df.reset_index().groupby('customer_id').agg({
    'sales': [min, max, sum, groupby_mean, groupby_count],
    'invoice_date': [min, max, purchase_duration, avg_frequency]
})

In [ ]:
summary_df

In [ ]:
summary_df.columns = ['_'.join(col).lower() for col in summary_df.columns]

In [ ]:
summary_df

In [ ]:
summary_df.shape

In [ ]:
summary_df = summary_df.loc[summary_df['invoice_date_purchase_duration'] > 0]

In [ ]:
summary_df.shape

In [ ]:
import plotly.express as px

plot_df = (
    summary_df.groupby('sales_count')
    .count()['sales_avg']
    .iloc[:20]
    .reset_index(name='count')
)

fig = px.bar(
    plot_df,
    x='sales_count',
    y='count',
    template='ggplot2',
    color_discrete_sequence=['skyblue'],
    width=1200,
    height=700
)

fig.update_layout(
    xaxis_title='sales_count',
    yaxis_title='count'
)

fig.show()

In [ ]:
summary_df['sales_count'].describe()

In [ ]:
summary_df['sales_avg'].describe()

In [ ]:
fig = px.histogram(
    summary_df,
    x='invoice_date_purchase_frequency',
    nbins=20,
    template='ggplot2',
    color_discrete_sequence=['skyblue'],
    width=1200,
    height=700
)

fig.update_layout(
    xaxis_title='avg. number of days between purchases',
    yaxis_title='count'
)

fig.show()

In [ ]:
summary_df['invoice_date_purchase_frequency'].describe()

In [ ]:
summary_df['invoice_date_purchase_duration'].describe()

# Predicting 3-Month CLV

## Data Preparation

In [ ]:
clv_freq = '3ME'

In [ ]:
data_df = orders_df.reset_index().groupby([
    'customer_id',
    pd.Grouper(key='invoice_date', freq=clv_freq)
]).agg({
    'sales': [sum, groupby_mean, groupby_count],
})

In [ ]:
data_df.columns = ['_'.join(col).lower() for col in data_df.columns]

In [ ]:
data_df = data_df.reset_index()

In [ ]:
data_df.head(10)

In [ ]:
date_month_map = {
    str(x)[:10]: 'M_%s' % (i+1) for i, x in enumerate(
        sorted(data_df.reset_index()['invoice_date'].unique(), reverse=True)
    )
}

In [ ]:
data_df['M'] = data_df['invoice_date'].apply(lambda x: date_month_map[str(x)[:10]])

In [ ]:
date_month_map

In [ ]:
data_df.head(10)

#### - Building Sample Set

In [ ]:
features_df = pd.pivot_table(
    data_df.loc[data_df['M'] != 'M_1'], 
    values=['sales_sum', 'sales_avg', 'sales_count'], 
    columns='M', 
    index='customer_id'
)

In [ ]:
features_df.columns = ['_'.join(col) for col in features_df.columns]

In [ ]:
features_df.shape

In [ ]:
features_df.head(10)

In [ ]:
features_df = features_df.fillna(0)

In [ ]:
features_df.head()

In [ ]:
response_df = data_df.loc[
    data_df['M'] == 'M_1',
    ['customer_id', 'sales_sum']
]

In [ ]:
response_df.columns = ['customer_id', 'clv_'+clv_freq]

In [ ]:
response_df.shape

In [ ]:
response_df.head(10)

In [ ]:
sample_set_df = features_df.merge(
    response_df, 
    left_index=True, 
    right_on='customer_id',
    how='left'
)

In [ ]:
response_idx = response_df.set_index('customer_id')
sample_set_df = features_df.merge(response_idx, left_index=True, right_index=True, how='left')

In [ ]:
sample_set_df.shape

In [ ]:
sample_set_df.head(10)

In [ ]:
sample_set_df = sample_set_df.fillna(0)

In [ ]:
sample_set_df.head()

In [ ]:
sample_set_df['clv_'+clv_freq].describe()

## Regression Models

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
target_var = 'clv_'+clv_freq
all_features = [x for x in sample_set_df.columns if x not in ['customer_id', target_var]]

In [ ]:
all_features

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    sample_set_df[all_features], 
    sample_set_df[target_var], 
    test_size=0.3,
    random_state=42
)

#### - Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression

# Try these models as well
# from sklearn.svm import SVR
# from sklearn.ensemble import RandomForestRegressor

In [ ]:
reg_fit = LinearRegression()

In [ ]:
reg_fit.fit(x_train, y_train)

In [ ]:
reg_fit.intercept_

In [ ]:
coef = pd.DataFrame(list(zip(all_features, reg_fit.coef_)))
coef.columns = ['feature', 'coef']

coef

## Evaluation

In [ ]:
from sklearn.metrics import r2_score, median_absolute_error

In [ ]:
train_preds =  reg_fit.predict(x_train)
test_preds = reg_fit.predict(x_test)

#### - R-Squared

In [ ]:
print('In-Sample R-Squared: %0.4f' % r2_score(y_true=y_train, y_pred=train_preds))
print('Out-of-Sample R-Squared: %0.4f' % r2_score(y_true=y_test, y_pred=test_preds))

#### - Median Absolute Error

In [ ]:
print('In-Sample MSE: %0.4f' % median_absolute_error(y_true=y_train, y_pred=train_preds))
print('Out-of-Sample MSE: %0.4f' % median_absolute_error(y_true=y_test, y_pred=test_preds))

#### - Scatter Plot

In [ ]:
fig = px.scatter(
    x=y_train,
    y=train_preds,
    template='ggplot2',
    labels={'x': 'actual', 'y': 'predicted'},
    title='In-Sample Actual vs. Predicted'
)

fig.add_shape(
    type='line',
    x0=0, y0=0,
    x1=max(y_train), y1=max(train_preds),
    line=dict(color='gray', width=1, dash='dash')
)

fig.update_layout(width=900, height=600)
fig.show()

In [ ]:
fig = px.scatter(
    x=y_test,
    y=test_preds,
    template='ggplot2',
    labels={'x': 'actual', 'y': 'predicted'},
    title='Out-of-Sample Actual vs. Predicted'
)

fig.add_shape(
    type='line',
    x0=0, y0=0,
    x1=float(y_test.max()), y1=float(test_preds.max()),
    line=dict(color='gray', width=1, dash='dash')
)

fig.update_layout(width=900, height=600)
fig.show()